In [0]:
pip install mlxtend

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 132.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.7/32.7 MB 131.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Not uninstalling numpy at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-bda80bfd-50f1-4be3-a35a-b13002cfd713
    Can't uninstall 'numpy'. No files were found to uninstall.
  Attempting uninstall: joblib
    Found existing installation: joblib 1.4.2
    Not uninstalling joblib at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/

In [0]:

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, apriori, association_rules

# Load your data ---
grocery_data = pd.read_csv("/Volumes/workspace/ada_data/data_ada/grocery_data.csv")  
print(grocery_data.head())




   Member_number        Date itemDescription
0          22404  28-07-2014       pip fruit
1          22404  28-07-2014       chocolate
2          33845  27-03-2014          onions
3          33845  27-03-2014  tropical fruit
4          44517  12-11-2015     frankfurter


In [0]:
# Build a transaction ID: (Member_number, Date) = one shopping basket
grocery_data["txn_id"] = grocery_data["Member_number"].astype(str) + "_" + grocery_data["Date"].astype(str)

# Group items per transaction
transactions = (
    grocery_data.groupby("txn_id")["itemDescription"]
      .apply(lambda x: list(pd.unique(x)))  # unique items per basket
      .tolist()
)

print("Number of transactions:", len(transactions))
print("Example transaction:", transactions[0][:10])

Number of transactions: 205
Example transaction: ['frankfurter', 'bottled water']


In [0]:
# One-hot encode (tabular format) ---
te = TransactionEncoder()
X = te.fit(transactions).transform(transactions)  # boolean array
basket = pd.DataFrame(X, columns=te.columns_)
print(basket.head())

   Instant food products  UHT-milk  ...  white wine  yogurt
0                  False     False  ...       False   False
1                  False     False  ...       False   False
2                  False     False  ...       False   False
3                  False     False  ...       False   False
4                  False     False  ...       False   False

[5 rows x 103 columns]


In [0]:
# Specify the thresholds
min_sup = 0.01   # change as you like
min_conf = 0.3   # change as you like

In [0]:
# Apriori: frequent itemsets ---

freq_ap = apriori(basket, min_support=min_sup, use_colnames=True, max_len=3)  # max_len helps speed
rules_ap = association_rules(freq_ap, metric="confidence", min_threshold=min_conf)

print("\nApriori frequent itemsets (top 10):")
print(freq_ap.sort_values("support", ascending=False).head(10))

print("\nApriori rules (top 10 by confidence):")
print(rules_ap.sort_values(["confidence", "support"], ascending=False).head(10))


Apriori frequent itemsets (top 10):
     support                         itemsets
29  0.185366                frozenset({milk})
40  0.126829          frozenset({rolls/buns})
53  0.107317              frozenset({yogurt})
34  0.102439  frozenset({organic vegetables})
45  0.087805                frozenset({soda})
49  0.082927      frozenset({tropical fruit})
18  0.063415         frozenset({frankfurter})
42  0.058537             frozenset({sausage})
3   0.058537        frozenset({bottled beer})
4   0.053659       frozenset({bottled water})

Apriori rules (top 10 by confidence):
                       antecedents  ... kulczynski
1        frozenset({yogurt, soda})  ...   0.576923
2  frozenset({yogurt, rolls/buns})  ...   0.611111
3    frozenset({rolls/buns, soda})  ...   0.490909
0                frozenset({pork})  ...   0.189474

[4 rows x 14 columns]


In [0]:
# FP-Growth: frequent itemsets ---
freq_fp = fpgrowth(basket, min_support=min_sup, use_colnames=True)

# FP-Growth rules
rules_fp = association_rules(freq_fp, metric="confidence", min_threshold=min_conf)

print("\nFP-Growth frequent itemsets (top 10):")
print(freq_fp.sort_values("support", ascending=False).head(10))

print("\nFP-Growth rules (top 10 by confidence):")
print(rules_fp.sort_values(["confidence", "support"], ascending=False).head(10))




FP-Growth frequent itemsets (top 10):
     support                         itemsets
10  0.185366                frozenset({milk})
3   0.126829          frozenset({rolls/buns})
14  0.107317              frozenset({yogurt})
11  0.102439  frozenset({organic vegetables})
16  0.087805                frozenset({soda})
30  0.082927      frozenset({tropical fruit})
0   0.063415         frozenset({frankfurter})
26  0.058537        frozenset({bottled beer})
4   0.058537             frozenset({sausage})
37  0.053659  frozenset({whipped/sour cream})

FP-Growth rules (top 10 by confidence):
                       antecedents  ... kulczynski
0  frozenset({yogurt, rolls/buns})  ...   0.611111
1        frozenset({yogurt, soda})  ...   0.576923
2    frozenset({soda, rolls/buns})  ...   0.490909
3                frozenset({pork})  ...   0.189474

[4 rows x 14 columns]
